In [0]:
# Cargar credenciales de LSEG desde Databricks Secrets
APP_KEY = dbutils.secrets.get(scope="refinitiv_scope", key="app_key")
USERNAME = dbutils.secrets.get(scope="refinitiv_scope", key="username")
PASSWORD = dbutils.secrets.get(scope="refinitiv_scope", key="password")

print("✓ Credenciales cargadas")

# PortfolioDownloader - Nueva Clase para Múltiples Instrumentos

He creado la nueva clase `PortfolioDownloader` en `analisis_precios_lseg/data/portfolio.py`.

## Características:

1. **Input**: Lista de RICs, rango de fechas, y periodicidad
2. **Output**: DataFrame con:
   - Fechas como índice
   - Cada RIC como nombre de columna
   - Solo precios de cierre
   - Full join (outer join) entre todos los instrumentos
   - Valores faltantes rellenados con el último precio conocido (forward fill)

3. **Métodos adicionales**:
   - `get_returns()`: Calcula retornos simples o logarítmicos
   - `get_correlation_matrix()`: Matriz de correlación entre instrumentos
   - `summary()`: Resumen estadístico del portafolio

## Ejemplo de uso:

Ver las celdas siguientes para ejemplos de cómo usar la nueva clase. 

In [0]:
%pip install lseg-data --quiet

import sys
sys.path.append('/Workspace/Users/nvaldez@tec.mx/finanzas_2026_agosto')

# Force reload of modules
import importlib
import analisis_precios_lseg.data.downloader
import analisis_precios_lseg.data.portfolio
importlib.reload(analisis_precios_lseg.data.downloader)
importlib.reload(analisis_precios_lseg.data.portfolio)

# Import after reload
from analisis_precios_lseg.data.portfolio import PortfolioDownloader

# Definir una lista de RICs para el portafolio
rics = ['AAPL.O', 'MSFT.O', 'GOOGL.O', 'AMZN.O', 'TSLA.O']

# Crear el downloader de portafolio
portfolio = PortfolioDownloader(
    rics=rics,
    app_key=APP_KEY,
    username=USERNAME,
    password=PASSWORD,
    use_secrets=False
)

# Descargar datos de los últimos 180 días
data = portfolio.download_portfolio(days=180)

# Calcular retornos diarios
returns = portfolio.get_returns(log_returns=False)

print("Retornos diarios (primeras 10 filas):")
print(returns.head(10))

# Calcular matriz de correlación
print("\n" + "="*60)
print("MATRIZ DE CORRELACIÓN ENTRE INSTRUMENTOS")
print("="*60)
corr_matrix = portfolio.get_correlation_matrix()
print(corr_matrix)

# Visualizar la matriz de correlación
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, 
            square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Matriz de Correlación del Portafolio', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [0]:
import sys
sys.path.append('/Workspace/Users/nvaldez@tec.mx/finanzas_2026_agosto')

from analisis_precios_lseg.data import PortfolioDownloader

# Definir credenciales (estas deben estar cargadas previamente)
# APP_KEY, USERNAME, PASSWORD deben estar definidos

# Definir una lista de RICs para el portafolio
rics = ['AAPL.O', 'MSFT.O', 'GOOGL.O', 'AMZN.O', 'TSLA.O']

# Crear el downloader de portafolio
portfolio = PortfolioDownloader(
    rics=rics,
    app_key=APP_KEY,
    username=USERNAME,
    password=PASSWORD,
    use_secrets=False
)

# Descargar datos de los últimos 180 días
data = portfolio.download_portfolio(days=180)

# Mostrar las primeras filas
print("\nPrimeras 5 filas del portafolio:")
print(data.head())

# Mostrar las últimas filas
print("\nÚltimas 5 filas del portafolio:")
print(data.tail())

# Mostrar resumen estadístico
portfolio.summary()

In [0]:
data

In [0]:
import numpy as np
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize

# Calcular retornos diarios
returns = data.pct_change().dropna()

# Parámetros del modelo
mean_returns = returns.mean() * 252  # Anualizado
cov_matrix = returns.cov() * 252  # Anualizado
num_assets = len(data.columns)
risk_free_rate = 0.02  # 2% tasa libre de riesgo

print("="*70)
print("RETORNOS ESPERADOS ANUALIZADOS")
print("="*70)
for ticker, ret in mean_returns.items():
    print(f"{ticker:10s}: {ret*100:6.2f}%")

print("\n" + "="*70)
print("MATRIZ DE COVARIANZA ANUALIZADA")
print("="*70)
print(cov_matrix)

# Función para calcular retorno y riesgo del portafolio
def portfolio_performance(weights, mean_returns, cov_matrix):
    portfolio_return = np.sum(mean_returns * weights)
    portfolio_std = np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights)))
    return portfolio_return, portfolio_std

# Función para Sharpe Ratio negativo (para minimización)
def negative_sharpe(weights, mean_returns, cov_matrix, risk_free_rate):
    p_return, p_std = portfolio_performance(weights, mean_returns, cov_matrix)
    return -(p_return - risk_free_rate) / p_std

# Función para varianza del portafolio (para minimización)
def portfolio_variance(weights, mean_returns, cov_matrix):
    return portfolio_performance(weights, mean_returns, cov_matrix)[1]**2

# Restricciones y límites
constraints = ({'type': 'eq', 'fun': lambda x: np.sum(x) - 1})  # Suma = 1
bounds = tuple((0, 1) for _ in range(num_assets))  # Pesos entre 0 y 1
initial_weights = np.array([1/num_assets] * num_assets)

# 1. Encontrar el portafolio de mínima varianza
min_var_result = minimize(
    portfolio_variance,
    initial_weights,
    args=(mean_returns, cov_matrix),
    method='SLSQP',
    bounds=bounds,
    constraints=constraints
)
min_var_return, min_var_std = portfolio_performance(
    min_var_result.x, mean_returns, cov_matrix
)

# 2. Encontrar el portafolio de máximo Sharpe ratio
max_sharpe_result = minimize(
    negative_sharpe,
    initial_weights,
    args=(mean_returns, cov_matrix, risk_free_rate),
    method='SLSQP',
    bounds=bounds,
    constraints=constraints
)
max_sharpe_return, max_sharpe_std = portfolio_performance(
    max_sharpe_result.x, mean_returns, cov_matrix
)

print("\n" + "="*70)
print("PORTAFOLIOS ÓPTIMOS")
print("="*70)

print("\n1. PORTAFOLIO DE MÍNIMA VARIANZA:")
print(f"   Retorno esperado: {min_var_return*100:.2f}%")
print(f"   Volatilidad:      {min_var_std*100:.2f}%")
print("   Asignación:")
for ticker, weight in zip(data.columns, min_var_result.x):
    if weight > 0.01:  # Solo mostrar si > 1%
        print(f"      {ticker:10s}: {weight*100:5.2f}%")

print("\n2. PORTAFOLIO DE MÁXIMO SHARPE RATIO:")
print(f"   Retorno esperado: {max_sharpe_return*100:.2f}%")
print(f"   Volatilidad:      {max_sharpe_std*100:.2f}%")
print(f"   Sharpe Ratio:     {(max_sharpe_return - risk_free_rate)/max_sharpe_std:.3f}")
print("   Asignación:")
for ticker, weight in zip(data.columns, max_sharpe_result.x):
    if weight > 0.01:  # Solo mostrar si > 1%
        print(f"      {ticker:10s}: {weight*100:5.2f}%")

print("\n" + "="*70)

# ==================== GENERAR LA FRONTERA EFICIENTE ====================
print("\nGenerando frontera eficiente...")
num_portfolios = 50
target_returns = np.linspace(min_var_return, mean_returns.max(), num_portfolios)

efficient_portfolios = []

for target_return in target_returns:
    # Restricción adicional: retorno objetivo
    constraints = (
        {'type': 'eq', 'fun': lambda x: np.sum(x) - 1},
        {'type': 'eq', 'fun': lambda x: portfolio_performance(x, mean_returns, cov_matrix)[0] - target_return}
    )
    
    result = minimize(
        portfolio_variance,
        initial_weights,
        args=(mean_returns, cov_matrix),
        method='SLSQP',
        bounds=bounds,
        constraints=constraints
    )
    
    if result.success:
        p_return, p_std = portfolio_performance(result.x, mean_returns, cov_matrix)
        efficient_portfolios.append({
            'return': p_return,
            'std': p_std,
            'sharpe': (p_return - risk_free_rate) / p_std,
            'weights': result.x
        })

# Convertir a DataFrame
ef_df = pd.DataFrame(efficient_portfolios)

# Generar portafolios aleatorios para comparación
num_random = 5000
random_portfolios = []

np.random.seed(42)
for _ in range(num_random):
    weights = np.random.random(num_assets)
    weights /= np.sum(weights)
    p_return, p_std = portfolio_performance(weights, mean_returns, cov_matrix)
    random_portfolios.append({
        'return': p_return,
        'std': p_std,
        'sharpe': (p_return - risk_free_rate) / p_std
    })

random_df = pd.DataFrame(random_portfolios)

# Crear figura con 2 subplots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))

# ========== SUBPLOT 1: FRONTERA EFICIENTE ==========
ax1.scatter(random_df['std']*100, random_df['return']*100, 
           c=random_df['sharpe'], cmap='YlOrRd', alpha=0.3, s=10,
           label='Portafolios Aleatorios')

ax1.plot(ef_df['std']*100, ef_df['return']*100, 
        'b-', linewidth=3, label='Frontera Eficiente')

ax1.scatter(min_var_std*100, min_var_return*100, 
           marker='*', color='green', s=500, edgecolors='black', linewidth=2,
           label='Mínima Varianza', zorder=5)

ax1.scatter(max_sharpe_std*100, max_sharpe_return*100, 
           marker='*', color='red', s=500, edgecolors='black', linewidth=2,
           label='Máximo Sharpe', zorder=5)

ax1.set_xlabel('Volatilidad Anualizada (%)', fontsize=12, fontweight='bold')
ax1.set_ylabel('Retorno Esperado Anualizado (%)', fontsize=12, fontweight='bold')
ax1.set_title('Frontera Eficiente de Markowitz', fontsize=14, fontweight='bold')
ax1.legend(loc='best', fontsize=10)
ax1.grid(True, alpha=0.3)

cbar = plt.colorbar(ax1.collections[0], ax=ax1)
cbar.set_label('Sharpe Ratio', fontsize=10)

# ========== SUBPLOT 2: ASSET ALLOCATION AREA CHART ==========
weights_array = np.array([p['weights'] for p in efficient_portfolios])
risk_levels = ef_df['std'].values * 100

ax2.stackplot(risk_levels, 
             weights_array.T * 100,
             labels=data.columns,
             alpha=0.8)

ax2.set_xlabel('Volatilidad del Portafolio (%)', fontsize=12, fontweight='bold')
ax2.set_ylabel('Asignación de Activos (%)', fontsize=12, fontweight='bold')
ax2.set_title('Asignación de Activos a lo Largo de la Frontera Eficiente', 
             fontsize=14, fontweight='bold')
ax2.legend(loc='upper left', fontsize=10, bbox_to_anchor=(1.05, 1))
ax2.grid(True, alpha=0.3, axis='y')
ax2.set_ylim([0, 100])

# Marcar puntos clave
min_var_idx = np.argmin(np.abs(ef_df['std'].values - min_var_std))
max_sharpe_idx = np.argmin(np.abs(ef_df['std'].values - max_sharpe_std))

ax2.axvline(x=risk_levels[min_var_idx], color='green', 
           linestyle='--', linewidth=2, alpha=0.7, label='Mín. Varianza')
ax2.axvline(x=risk_levels[max_sharpe_idx], color='red', 
           linestyle='--', linewidth=2, alpha=0.7, label='Máx. Sharpe')

plt.tight_layout()
plt.show()

print("\n" + "="*70)
print("ANÁLISIS DE LA FRONTERA EFICIENTE")
print("="*70)
print(f"\nNúmero de portafolios en la frontera: {len(efficient_portfolios)}")
print(f"Rango de volatilidad: {ef_df['std'].min()*100:.2f}% - {ef_df['std'].max()*100:.2f}%")
print(f"Rango de retorno: {ef_df['return'].min()*100:.2f}% - {ef_df['return'].max()*100:.2f}%")
print(f"Sharpe Ratio máximo: {ef_df['sharpe'].max():.3f}")